## KPI & Visualization

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

with open("../config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

ship = pd.read_csv("../data/processed/shipments_simulated.csv")
lanes = pd.read_csv("../data/processed/lanes_generated.csv")

ship.head()

In [ ]:
expected_base_days = lanes["base_lead_days"].mean()
expected_base_days

In [ ]:
ship["expected_lead_days"] = expected_base_days
ship["delay_days"] = (ship["realized_lead_days"] - ship["expected_lead_days"]).clip(lower=0)
ship["cost_per_unit"] = ship["transport_cost"] / ship["demand_units"].replace(0, np.nan)
ship["on_time_flag"] = ship["delivered_on_time"].astype(int)

ship.describe(include="all").T.head(12)

In [ ]:
def kpi_agg(df):
    return pd.Series({
        "shipments": len(df),
        "units": df["demand_units"].sum(),
        "otd_pct": df["on_time_flag"].mean() * 100,
        "avg_lead_days": df["realized_lead_days"].mean(),
        "avg_delay_days": df["delay_days"].mean(),
        "total_cost": df["transport_cost"].sum(),
        "cost_per_unit_avg": (df["transport_cost"].sum() / max(df["demand_units"].sum(), 1))
    })

In [ ]:
kpi_overall = kpi_agg(ship)
kpi_by_week = ship.groupby("week").apply(kpi_agg).reset_index()
kpi_by_retailer = ship.groupby("retailer_id").apply(kpi_agg).reset_index()
kpi_by_sku = ship.groupby("sku_id").apply(kpi_agg).reset_index()

kpi_overall, kpi_by_week.head(), kpi_by_retailer.head(), kpi_by_sku.head()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(kpi_by_week["week"], kpi_by_week["otd_pct"], marker="o")
plt.title("On-Time Delivery % by Week")
plt.xlabel("Week"); plt.ylabel("OTD %"); plt.grid(alpha=0.3); plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(kpi_by_week["week"], kpi_by_week["avg_lead_days"], marker="o")
plt.axhline(expected_base_days, linestyle="--")
plt.title("Average Lead Time by Week")
plt.xlabel("Week"); plt.ylabel("Days"); plt.grid(alpha=0.3); plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(kpi_by_retailer["retailer_id"], kpi_by_retailer["total_cost"])
plt.title("Total Transport Cost by Retailer")
plt.xlabel("Retailer"); plt.ylabel("Total Cost"); plt.grid(axis="y", alpha=0.3); plt.show()

In [ ]:
out_dir = Path("../data/processed"); out_dir.mkdir(parents=True, exist_ok=True)
kpi_by_week.to_csv(out_dir/"kpi_by_week.csv", index=False)
kpi_by_retailer.to_csv(out_dir/"kpi_by_retailer.csv", index=False)
kpi_by_sku.to_csv(out_dir/"kpi_by_sku.csv", index=False)

summary = pd.DataFrame([kpi_overall]).round(2)
summary.to_csv(out_dir/"kpi_summary.csv", index=False)
print("✅ Saved KPI outputs to data/processed/")
summary